In [1]:
#!/usr/bin/env python3
"""
Scrape the NYCHA Outages History table and export to CSV using Playwright and pandas.

Usage:
  python scrape.py --output nycha_outages_history.csv

Notes:
  - Requires Python packages: playwright, pandas, lxml
  - After installing playwright, you must install a browser once:
	  python -m playwright install chromium
"""

from __future__ import annotations

import argparse
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError


URL = "https://my.nycha.info/Outages/Outages.aspx#tab_history"


def scrape_history_table(headless: bool = False, timeout_ms: int = 30000) -> pd.DataFrame:
	"""Navigate to the NYCHA Outages History tab and return its table as a DataFrame.

	Args:
		headless: Run the browser in headless mode.
		timeout_ms: Max time to wait for page/table loads in milliseconds.

	Returns:
		A pandas DataFrame containing the table rows.
	"""

	with sync_playwright() as p:
		browser = p.chromium.launch(
			headless=headless,
			executable_path = "/Applications/Google Chrome Canary.app/Contents/MacOS/Google Chrome Canary")
		context = browser.new_context()
		page = context.new_page()

		# Navigate and wait until network is idle to reduce flakiness
		page.goto(URL, wait_until="domcontentloaded", timeout=timeout_ms)

		# Ensure the History tab is activated (anchor + possible JS event)
		try:
			# Some sites require an explicit click to trigger data-binding
			history_tab_selector = 'a[href="#tab_history"], button[data-bs-target="#tab_history"], button[aria-controls="tab_history"]'
			page.locator(history_tab_selector).first.click(timeout=5000)
		except PlaywrightTimeoutError:
			# If not found, the hash may have already activated the tab; continue
			pass

		# Wait for the history tab content to be visible
		tab_panel = page.locator('#tab_history')
		try:
			tab_panel.wait_for(state="visible", timeout=timeout_ms)
		except PlaywrightTimeoutError:
			# As a fallback, the panel may not toggle visibility; proceed
			pass

		# If there's a page-length selector, try to switch to "All" to capture all rows.
		try:
			length_select = tab_panel.locator('select')
			if length_select.count() > 0:
				# Try selecting the option labeled "All" if present
				try:
					length_select.first.select_option(label="All")
					# Give the table a moment to re-render
					page.wait_for_timeout(800)
				except Exception:
					# If no "All" option, try the largest numeric option
					try:
						options = length_select.first.locator('option').all_text_contents()
						numeric = [int(o.strip()) for o in options if o.strip().isdigit()]
						if numeric:
							max_val = str(max(numeric))
							length_select.first.select_option(label=max_val)
							page.wait_for_timeout(500)
					except Exception:
						pass
		except Exception:
			pass

		# Wait for the table and for at least one row to appear
		table = tab_panel.locator('table')
		table.wait_for(state="attached", timeout=timeout_ms)
		try:
			tab_panel.locator('table tbody tr').first.wait_for(state="visible", timeout=timeout_ms)
		except PlaywrightTimeoutError:
			# Sometimes rows render but are not individually visible; proceed
			pass

		# Extract the table's outer HTML and parse with pandas
		table_html = table.evaluate("(el) => el.outerHTML")

		browser.close()

	# Use pandas to parse the HTML table. Requires lxml or html5lib backend.
	dfs = pd.read_html(table_html)  # type: ignore[arg-type]
	if not dfs:
		raise RuntimeError("No tables were found in the history tab HTML")
	df = dfs[0]
	return df


def main(argv: list[str] | None = None) -> int:
	parser = argparse.ArgumentParser(description="Scrape NYCHA Outages History table to CSV")
	parser.add_argument(
		"--output",
		"-o",
		type=Path,
		help="Output CSV file path (default: ./nycha_outages_history_YYYYMMDD.csv)",
	)
	parser.add_argument(
		"--headful",
		action="store_true",
		help="Run the browser in headful (non-headless) mode for debugging",
	)
	parser.add_argument(
		"--timeout",
		type=int,
		default=30,
		help="Timeout in seconds for page/table loads (default: 30)",
	)

	args = parser.parse_args(argv)
	out_path: Path
	if args.output is None:
		stamp = datetime.now().strftime("%Y%m%d")
		out_path = Path(f"nycha_outages_history_{stamp}.csv")
	else:
		out_path = args.output
	out_path.parent.mkdir(parents=True, exist_ok=True)

	try:
		df = scrape_history_table(headless=not args.headful, timeout_ms=args.timeout * 1000)
	except Exception as e:
		print(f"Error: {e}", file=sys.stderr)
		return 2

	# Basic cleanup: strip surrounding whitespace in string columns
	try:
		for col in df.select_dtypes(include=["object"]).columns:
			df[col] = df[col].astype(str).str.strip()
	except Exception:
		pass

	df.to_csv(out_path, index=False)
	print(f"Wrote {len(df):,} rows to {out_path}")
	return 0


if __name__ == "__main__":
	raise SystemExit(main())


usage: ipykernel_launcher.py [-h] [--output OUTPUT] [--headful]
                             [--timeout TIMEOUT]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/amandafernandez/Library/Jupyter/runtime/kernel-v3b358aeb1d222a5f82a0a1ab23ce9cd5072d7c722.json


SystemExit: 2

/Users/amandafernandez/.pyenv/versions/3.13.9/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
